In [ ]:
from pathlib import Path
from pynq import Overlay
from pynq import allocate
import numpy as np

overlay_dir = Path("../bit_hwh_files").resolve()
slowOverlay = Overlay(str(overlay_dir / "slowMult.bit"))

print("available IPs:", slowOverlay.ip_dict.keys())
print("Overlay loaded.")

slowMultIP = slowOverlay.matrixMult_0


In [ ]:
#get the register map
print(slowMultIP.register_map)


In [ ]:
N = 16
# allocate memory for 3 16x16 matrices stored in ddr
A = allocate(shape=(N, N), dtype=np.int32)
B = allocate(shape=(N, N), dtype=np.int32)
C = allocate(shape=(N, N), dtype=np.int32)


# initializes A and B with random integers
A[:] = np.random.randint(-5, 5, (N, N))
B[:] = np.random.randint(-5, 5, (N, N))

print(A, "\n\n", B)
#print(B)

# save IP's AXI-Lite Registers
regMap = slowMultIP.register_map

# mask in addresses for A,B,C
# needs to be masked because CPU uses 64 bit addressing, but AXI-lite is only 32 bit wide
regMap.A_1.A = A.physical_address & 0xFFFFFFFF
regMap.A_2.A = (A.physical_address >> 32) & 0xFFFFFFFF

regMap.B_1.B = B.physical_address & 0xFFFFFFFF
regMap.B_2.B = (B.physical_address >> 32) & 0xFFFFFFFF

regMap.C_1.C = C.physical_address & 0xFFFFFFFF
regMap.C_2.C = (C.physical_address >> 32) & 0xFFFFFFFF


In [ ]:
from time import perf_counter

# record start time
startTime = perf_counter()

# starts Matrix Accelerator
slowMultIP.register_map.CTRL.AP_START = 1

# wait until FPGA finishes (poll the done bit)
while slowMultIP.register_map.CTRL.AP_DONE == 0:
    pass

# record end time and calculate total time
endTime = perf_counter()
totalTime = endTime - startTime

print("Hardware multiply finished!")

In [ ]:
# create the baseline reference and compare results
C_ref = A @ B

print(C_ref, "\n\n", C)
print("Max error:", np.max(np.abs(C_ref - C)))
print("Correct:", np.array_equal(C_ref, C))


In [ ]:
def numpyVSaccel( ip, A, B, C, nTests=50 ):
    numpy_times = []
    accel_times = []
    all_correct = True
    for i in range(nTests):
        # random test case
        A_np = np.random.randint(-100, 100, (N, N))
        B_np = np.random.randint(-100, 100, (N, N))
        
        # NumPy timing
        startTime = perf_counter()
        C_ref = A_np @ B_np
        endTime = perf_counter()
        numpy_times.append(endTime - startTime)
        
        # copy to FPGA buffers
        A[:] = A_np
        B[:] = B_np
        
        # run accelerator
        regMap = ip.register_map

        regMap.A_1.A = A.physical_address & 0xFFFFFFFF
        regMap.A_2.A = (A.physical_address >> 32) & 0xFFFFFFFF

        regMap.B_1.B = B.physical_address & 0xFFFFFFFF
        regMap.B_2.B = (B.physical_address >> 32) & 0xFFFFFFFF

        regMap.C_1.C = C.physical_address & 0xFFFFFFFF
        regMap.C_2.C = (C.physical_address >> 32) & 0xFFFFFFFF
        
        # accelerator timing
        startTime = perf_counter()
        ip.register_map.CTRL.AP_START = 1
        while ip.register_map.CTRL.AP_DONE == 0:
            pass
        endTime = perf_counter()
        accel_times.append(endTime - startTime)
        
        # verify
        if not np.array_equal(C_ref, C):
            all_correct = False
            print(f"ERROR: Mismatch on trial {i}")
            break
        
    # print the metrics
    print("Correct for all trials:", all_correct)
    print(f"Avg NumPy time:   {np.mean(numpy_times)*1e6:.2f} us")
    print(f"Avg Accel time:   {np.mean(accel_times)*1e6:.2f} us")
    print(f"Speedup:          {np.mean(numpy_times) / np.mean(accel_times):.2f}x")

    return numpy_times, accel_times

In [ ]:
# slow results
print("Slow Matrix Multiplication Implementation")
numpy_times_slow, accel_times_slow = numpyVSaccel(slowMultIP, A, B, C, 50)

In [ ]:
# load fast overlay
fastOverlay = Overlay(str(overlay_dir / "fastMult.bit"))
print("available IPs:", fastOverlay.ip_dict.keys())
fastMultIP = fastOverlay.matrixMultFast_0

#get the register map
#print(slowMultIP.register_map)

# fast results
print("Fast Matrix Multiplication Implementation")
numpy_times_fast, accel_times_fast = numpyVSaccel(fastMultIP, A, B, C, 50)